In [1]:
# GPU
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Torch version: 2.5.1+cu121
CUDA available: True
Device: cuda


In [2]:
# imports
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
import numpy as np
import pandas as pd

In [3]:
# Load Bio_ClinicalBERT
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# create a tiny synthetic clinical dataset
data = {
    "text": [
        "Patient presents with chest pain radiating to left arm and shortness of breath. ECG shows ST elevation.",
        "Post-operative patient with fever and elevated CRP. Wound site appears red and swollen. Possible infection.",
        "Elderly patient with history of COPD, increased breathlessness over 3 days, using inhaler more frequently.",
        "Patient reports mild headache and fatigue. Observations within normal range. Discharged with advice to rest.",
        "Patient attended A&E following a fall. X-ray confirms fractured wrist. Plaster applied, follow-up in fracture clinic.",
        "Patient experiencing dizziness and blurred vision. Blood pressure elevated. Possible hypertensive episode.",
        "Patient complains of sore throat and runny nose. No fever. Likely viral upper respiratory infection.",
        "Patient with abdominal pain, nausea, and vomiting. CT scan suggests possible appendicitis."
    ],
    "label": [
        1,  # MI suspicion
        1,  # post-op infection
        1,  # COPD exacerbation
        0,  # mild headache
        0,  # wrist fracture, stable
        1,  # hypertensive episode
        0,  # mild URTI
        1   # appendicitis suspicion
    ]
}

df = pd.DataFrame(data)
df

,text,label
0,Patient presents with chest pain radiating to ...,1
1,Post-operative patient with fever and elevated...,1
2,"Elderly patient with history of COPD, increase...",1
3,Patient reports mild headache and fatigue. Obs...,0
4,Patient attended A&E following a fall. X-ray c...,0
5,Patient experiencing dizziness and blurred vis...,1
6,Patient complains of sore throat and runny nos...,0
7,"Patient with abdominal pain, nausea, and vomit...",1


In [5]:
# train / validation split
train_df = df.sample(frac=0.75, random_state=42)
val_df = df.drop(train_df.index)

train_df, val_df

(                                                text  label
 1  Post-operative patient with fever and elevated...      1
 5  Patient experiencing dizziness and blurred vis...      1
 0  Patient presents with chest pain radiating to ...      1
 7  Patient with abdominal pain, nausea, and vomit...      1
 2  Elderly patient with history of COPD, increase...      1
 4  Patient attended A&E following a fall. X-ray c...      0,
                                                 text  label
 3  Patient reports mild headache and fatigue. Obs...      0
 6  Patient complains of sore throat and runny nos...      0)

In [6]:
# tokenisation function
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)

train_ds = train_ds.remove_columns(["text", "__index_level_0__"])
val_ds = val_ds.remove_columns(["text", "__index_level_0__"])

train_ds.set_format("torch")
val_ds.set_format("torch")


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [7]:
# metrics: accuracy, f1
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [8]:
# training arguments
training_args = TrainingArguments(
    output_dir="./clinicalbert-finetune",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    report_to="none"
)


C:\Users\marke\projects\data-science\nhs\.venv\Lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [9]:
# trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [10]:
# fine tune Bio_ClinicalBERT
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.643600,1.007141,0.000000,0.000000
2,0.429800,0.987722,0.000000,0.000000
3,0.455800,0.983264,0.000000,0.000000
4,0.345000,0.972023,0.000000,0.000000
5,0.369300,0.969035,0.000000,0.000000


TrainOutput(global_step=10, training_loss=0.4487214803695679, metrics={'train_runtime': 3.9275, 'train_samples_per_second': 7.638, 'train_steps_per_second': 2.546, 'total_flos': 1973332915200.0, 'train_loss': 0.4487214803695679, 'epoch': 5.0})

In [11]:
# evaluation on validation set
metrics = trainer.evaluate()
metrics

{'eval_loss': 0.9690350294113159,
 'eval_accuracy': 0.0,
 'eval_f1': 0.0,
 'eval_runtime': 0.0239,
 'eval_samples_per_second': 83.837,
 'eval_steps_per_second': 41.919,
 'epoch': 5.0}

In [12]:
# inference on new clinical set
def predict_severity(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    preds = np.argmax(probs, axis=-1)
    return preds, probs

test_texts = [
    "Patient presents with severe chest pain and sweating. ECG pending.",
    "Patient reports mild back pain after gardening. No red flag symptoms."
]

preds, probs = predict_severity(test_texts)

for t, p, pr in zip(test_texts, preds, probs):
    print("\nTEXT:", t)
    print("PRED LABEL:", pr)
    print("CLASS:", "HIGH SEVERITY" if p == 1 else "LOW/MODERATE")


TEXT: Patient presents with severe chest pain and sweating. ECG pending.
PRED LABEL: [0.3131093  0.68689066]
CLASS: HIGH SEVERITY

TEXT: Patient reports mild back pain after gardening. No red flag symptoms.
PRED LABEL: [0.38174444 0.61825556]
CLASS: HIGH SEVERITY
